# RAG (Retrieval-Augmented Generation) Implementation

This notebook demonstrates a simple RAG system that:
1. Retrieves relevant documents using cosine similarity
2. Generates responses using Ollama LLM

## Define Sample Query
We'll use a sample query to test our document retrieval system.

In [1]:
user_query = "i am a data scientist and i live in india"

## Define Sample Document
This is the document we'll match against our query.

In [2]:
document = "india is a country where you can find some of the great data scientist live"

## Import Required Libraries
Import the necessary Python libraries for text processing, math operations, and API calls.

In [ ]:
from collections import Counter
import math
import json
import requests
import re

## Tokenize the Query
Break down the query into individual words (tokens) and convert to lowercase for comparison.

In [ ]:
clean_query = re.sub(r'[^\w\s]', '', user_query)
query_tokens = clean_query.lower().split()
print("Query tokens:", query_tokens)

Query tokens: ['i', 'am', 'a', 'data', 'scientist', 'and', 'i', 'live', 'in', 'india']


## Tokenize the Document
Break down the document into individual words (tokens) and convert to lowercase.

In [ ]:
clean_doc = re.sub(r'[^\w\s]', '', document)
document_tokens = clean_doc.lower().split()
print("Document tokens:", document_tokens)

Document tokens: ['india', 'is', 'a', 'country', 'where', 'you', 'can', 'find', 'some', 'of', 'the', 'great', 'data', 'scientist', 'live']


## Count Token Frequencies in Query
Use Counter to count how many times each word appears in the query.

In [6]:
query_Counter = Counter(query_tokens)
print("Query token counts:", dict(query_Counter))

Query token counts: {'i': 2, 'am': 1, 'a': 1, 'data': 1, 'scientist': 1, 'and': 1, 'live': 1, 'in': 1, 'india': 1}


## Count Token Frequencies in Document
Use Counter to count how many times each word appears in the document.

In [7]:
document_Counter = Counter(document_tokens)
print("Document token counts:", dict(document_Counter))

Document token counts: {'india': 1, 'is': 1, 'a': 1, 'country': 1, 'where': 1, 'you': 1, 'can': 1, 'find': 1, 'some': 1, 'of': 1, 'the': 1, 'great': 1, 'data': 1, 'scientist': 1, 'live': 1}


## Find Common Tokens
Identify which words appear in both the query and the document.

In [8]:
common_tokens = query_Counter.keys() & document_Counter.keys()
print("Common tokens:", list(common_tokens))

Common tokens: ['live', 'data', 'a', 'scientist', 'india']


## Calculate Dot Product
Calculate the dot product by summing the product of matching token frequencies.

In [9]:
dotprod = sum(query_Counter[token] * document_Counter[token] for token in common_tokens)
print(f"Dot product: {dotprod}")

Dot product: 5


## Calculate Magnitudes
Calculate the Euclidean norm (magnitude) for both query and document vectors.

In [10]:
query_magnitude = math.sqrt(sum(query_Counter[token] ** 2 for token in query_Counter))
print(f"Query magnitude: {query_magnitude:.4f}")

document_magnitude = math.sqrt(sum(document_Counter[token] ** 2 for token in document_Counter))
print(f"Document magnitude: {document_magnitude:.4f}")

Query magnitude: 3.4641
Document magnitude: 3.8730


## Calculate Cosine Similarity
Compute the cosine similarity score by dividing the dot product by the product of magnitudes.

In [11]:
similarity = dotprod / (query_magnitude * document_magnitude)
print(f"Cosine similarity: {similarity:.4f}")

Cosine similarity: 0.3727


## Define Document Corpus
Create a collection of documents that the RAG system will search through.

In [12]:
corpus_of_documents = [
    "Take a leisurely walk in the park and enjoy the fresh air.",
    "Visit a local museum and discover something new.",
    "Attend a live music concert and feel the rhythm.",
    "Go for a hike and admire the natural scenery.",
    "Have a picnic with friends and share some laughs.",
    "Explore a new cuisine by dining at an ethnic restaurant.",
    "Take a yoga class and stretch your body and mind.",
    "Join a local sports league and enjoy some friendly competition.",
    "Attend a workshop or lecture on a topic you're interested in.",
    "Visit an amusement park and ride the roller coasters."
]

print(f"Corpus contains {len(corpus_of_documents)} documents")

Corpus contains 10 documents


## Define Retrieval Function
Create a function that finds the most relevant document from the corpus based on cosine similarity with the query.

In [ ]:
def return_response(query, corpus_of_documents):
    clean_query = re.sub(r'[^\w\s]', '', query)
    query_tokens = clean_query.lower().split()
    query_Counter = Counter(query_tokens)
    
    # Calculate magnitude safely. If query has no valid tokens, magnitude is 0.
    query_sum_sq = sum(query_Counter[token] ** 2 for token in query_Counter)
    query_magnitude = math.sqrt(query_sum_sq) if query_sum_sq > 0 else 0
    
    best_similarity = -1
    best_document = None
    
    for document in corpus_of_documents:
        clean_doc = re.sub(r'[^\w\s]', '', document)
        document_tokens = clean_doc.lower().split()
        document_Counter = Counter(document_tokens)
        
        dotprod = sum(query_Counter[token] * document_Counter[token] 
                     for token in query_Counter.keys() & document_Counter.keys())
        
        doc_sum_sq = sum(document_Counter[token] ** 2 for token in document_Counter)
        document_magnitude = math.sqrt(doc_sum_sq) if doc_sum_sq > 0 else 0
        
        if query_magnitude > 0 and document_magnitude > 0:
            similarity = dotprod / (query_magnitude * document_magnitude)
        else:
            similarity = 0
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_document = document
    
    return best_document

## Test Retrieval - Query 1
Test the retrieval function with a query about fresh air.

In [14]:
query = "i like fresh air."
relevant_document = return_response(query, corpus_of_documents)
print(f"Query: '{query}'")
print(f"Retrieved document: '{relevant_document}'")

Query: 'i like fresh air.'
Retrieved document: 'Take a leisurely walk in the park and enjoy the fresh air.'


## Test Retrieval - Query 2
Test the retrieval function with a query about class.

In [15]:
user_input = "class"
relevant_document = return_response(user_input, corpus_of_documents)
print(f"Query: '{user_input}'")
print(f"Retrieved document: '{relevant_document}'")

Query: 'class'
Retrieved document: 'Take a yoga class and stretch your body and mind.'


In [16]:
query_sports = "I enjoy playing sports and competition"
relevant_doc_sports = return_response(query_sports, corpus_of_documents)
print(f"Query: '{query_sports}'")
print(f"Retrieved document: '{relevant_doc_sports}'")

Query: 'I enjoy playing sports and competition'
Retrieved document: 'Join a local sports league and enjoy some friendly competition.'


## Test Retrieval - Query 6
Test with a query about sports and physical activity.

In [17]:
query_food = "I want to try new food and restaurants"
relevant_doc_food = return_response(query_food, corpus_of_documents)
print(f"Query: '{query_food}'")
print(f"Retrieved document: '{relevant_doc_food}'")

Query: 'I want to try new food and restaurants'
Retrieved document: 'Take a yoga class and stretch your body and mind.'


## Test Retrieval - Query 5
Test with a query about food and dining.

In [18]:
query_music = "I love listening to music and concerts"
relevant_doc_music = return_response(query_music, corpus_of_documents)
print(f"Query: '{query_music}'")
print(f"Retrieved document: '{relevant_doc_music}'")

Query: 'I love listening to music and concerts'
Retrieved document: 'Attend a live music concert and feel the rhythm.'


## Test Retrieval - Query 4
Test with a query about music and entertainment.

In [19]:
query_hiking = "I want to go hiking in nature"
relevant_doc_hiking = return_response(query_hiking, corpus_of_documents)
print(f"Query: '{query_hiking}'")
print(f"Retrieved document: '{relevant_doc_hiking}'")

Query: 'I want to go hiking in nature'
Retrieved document: 'Go for a hike and admire the natural scenery.'


## Test Retrieval - Query 3
Test with a query about hiking and nature.

## Define Prompt Template
Create a template that instructs the LLM how to generate recommendations based on the retrieved document and user input.

In [20]:
prompt = """
You are a bot that makes recommendations for activities and lifestyle changes.
You answer only in 50 words making sure you encourage the user.
This is the recommended activity: {relevant_document}
The user input is: {user_input}
Compile a recommendation to the user based on the recommended activity
and the user input.
"""

## Call Ollama API
Send a request to the Ollama API with the formatted prompt to generate a personalized recommendation.

In [21]:
url = "http://localhost:11434/api/generate"

formatted_prompt = prompt.format(
    user_input=user_input,
    relevant_document=relevant_document
)

data = {
    "model": "gemma3:1b",
    "prompt": formatted_prompt
}

headers = {"Content-Type": "application/json"}

print("=" * 60)
print("Sending request to Ollama API...")
print(f"Model: {data['model']}")
print(f"User Input: {user_input}")
print(f"Retrieved Document: {relevant_document}")
print("=" * 60)
print("\nGenerating response...\n")

response = requests.post(
    url,
    data=json.dumps(data),
    headers=headers,
    stream=True
)

Sending request to Ollama API...
Model: gemma3:1b
User Input: class
Retrieved Document: Take a yoga class and stretch your body and mind.

Generating response...



## Process and Display Response
Collect the streaming response from Ollama and display the final recommendation.

In [22]:
full_response = []

for line in response.iter_lines():
    if line:
        decoded_line = json.loads(line.decode("utf-8"))
        response_text = decoded_line.get("response", "")
        full_response.append(response_text)

response.close()

final_response = "".join(full_response)
print(final_response)

Absolutely!  Taking a yoga class is a fantastic choice! It’s incredibly beneficial for your body and mind, reducing stress and boosting your energy.  It’s a gentle and rewarding way to feel centered and refreshed.  Start your wellness journey today! 😊
